In [14]:
from faker import Faker
import random
import sqlite3
import numpy as np
import math

from datetime import date, timedelta, datetime
from dateutil.relativedelta import relativedelta


# SETUP

conn = sqlite3.connect("Insurance_project.db")
cursor = conn.cursor()
fake = Faker()

END_DATE = date(2026, 4, 30)


# RESET TABLES

cursor.execute("DROP TABLE IF EXISTS Customers")
cursor.execute("DROP TABLE IF EXISTS Vehicle")
cursor.execute("DROP TABLE IF EXISTS Policy")
cursor.execute("DROP TABLE IF EXISTS Claims")


# CUSTOMERS

cursor.execute("""
CREATE TABLE Customers(
    CustomerID INTEGER PRIMARY KEY AUTOINCREMENT,
    Age INTEGER,
    Name TEXT,
    Location TEXT,
    Gender TEXT,
    DatePassedDrivingTest DATE
)
""")

def generate_person():
    gender = random.choice(["M", "F"])
    first = fake.first_name_male() if gender == "M" else fake.first_name_female()
    last = fake.last_name()
    age = int(np.random.beta(2, 5) * 62 + 18)
    return f"{first} {last}", gender, age


customer_data = []

for _ in range(10000):
    name, gender, age = generate_person()
    location = fake.city()

    start = END_DATE - relativedelta(years=age - 17)
    days = random.randint(0, max(1, (END_DATE - start).days))
    test_date = start + timedelta(days=days)

    customer_data.append((age, name, location, gender, test_date))

cursor.executemany("""
INSERT INTO Customers (Age, Name, Location, Gender, DatePassedDrivingTest)
VALUES (?, ?, ?, ?, ?)
""", customer_data)


# VEHICLES

cursor.execute("""
CREATE TABLE Vehicle(
    VehicleID INTEGER PRIMARY KEY AUTOINCREMENT,
    CustomerID INTEGER,
    Brand TEXT,
    Age INTEGER,
    Mileage INTEGER,
    Value INTEGER,
    FOREIGN KEY(CustomerID) REFERENCES Customers(CustomerID)
)
""")

brands = [
    "Ford","Toyota","Volkswagen","Honda","Nissan","Hyundai","Kia",
    "BMW","Mercedes-Benz","Audi","Peugeot","Renault","Mazda","Volvo",
    "Tesla","Jaguar","Land Rover","Porsche","Lexus","Ferrari","Lamborghini"
]

luxury = {"Ferrari","Lamborghini","Porsche"}
high = {"BMW","Mercedes-Benz","Audi","Jaguar","Tesla","Land Rover"}
medium = {"Honda","Nissan","Mazda","Volvo","Lexus","Toyota"}

vehicle_data = []

for customer_id in range(1, 10001):

    num_vehicles = random.choice([1, 1, 2])

    for _ in range(num_vehicles):

        age = random.randint(0, 15)
        annual_mileage = max(3000, int(np.random.normal(9000, 2500)))
        mileage = age * annual_mileage

        brand = random.choice(brands)

        if brand in luxury:
            base = random.randint(80000, 200000)
        elif brand in high:
            base = random.randint(30000, 70000)
        elif brand in medium:
            base = random.randint(15000, 35000)
        else:
            base = random.randint(8000, 20000)

        depreciation = math.exp(-0.00002 * mileage)
        value = max(2000, int(base * depreciation))

        vehicle_data.append((customer_id, brand, age, mileage, value))

cursor.executemany("""
INSERT INTO Vehicle (CustomerID, Brand, Age, Mileage, Value)
VALUES (?, ?, ?, ?, ?)
""", vehicle_data)


# POLICY

cursor.execute("""
CREATE TABLE Policy (
    PolicyID INTEGER PRIMARY KEY AUTOINCREMENT,
    CustomerID INTEGER,
    VehicleID INTEGER,
    StartDate DATE,
    EndDate DATE,
    LengthOfPolicy INTEGER,
    CoverageType TEXT,
    Excess REAL,
    Premium REAL
)
""")

coverage_types = ["Third party", "Third party, fire and theft", "Fully Comprehensive"]

coverage_risk = {
    "Third party": (0.03, 0.4),
    "Third party, fire and theft": (0.05, 0.6),
    "Fully Comprehensive": (0.08, 0.8)
}

def excess(age, value, coverage):
    base = 200 + 0.005 * value
    age_penalty = max(0, 30 - age) * 15

    modifier = {
        "Third party": 1.3,
        "Third party, fire and theft": 1.1,
        "Fully Comprehensive": 0.9
    }[coverage]

    return round(min(max((base + age_penalty) * modifier, 150), 1200), 2)


def premium(age, value, coverage, excess_val):
    freq, severity = coverage_risk[coverage]
    expected_loss = value * freq * severity

    age_penalty = max(0, 40 - age) * 5

    return round(max(250, min(
        (expected_loss * 1.5) + 200 + age_penalty - (0.2 * excess_val),
        6000
    )), 2)

policy_data = []

cursor.execute("SELECT VehicleID, CustomerID, Value FROM Vehicle")
vehicles = cursor.fetchall()

for vehicle_id, customer_id, value in vehicles:

    coverage = random.choices(coverage_types, weights=[0.3, 0.4, 0.3])[0]

    start = END_DATE - relativedelta(years=age - 17)
    duration = random.randint(1, 3)
    end = start + relativedelta(years=duration)

    age = random.randint(18, 80)

    ex = excess(age, value, coverage)
    prem = premium(age, value, coverage, ex)

    policy_data.append((
        customer_id,
        vehicle_id,
        start,
        end,
        duration,
        coverage,
        ex,
        prem
    ))

cursor.executemany("""
INSERT INTO Policy (
    CustomerID, VehicleID, StartDate, EndDate, LengthOfPolicy,
    CoverageType, Excess, Premium
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", policy_data)


# CLAIMS 

cursor.execute("""
CREATE TABLE Claims(
    ClaimsID INTEGER PRIMARY KEY AUTOINCREMENT,
    PolicyID INTEGER,
    ClaimDate DATE,
    ClaimType TEXT,
    ClaimStatus TEXT,
    Amount REAL,
    FOREIGN KEY (PolicyID) REFERENCES Policy(PolicyID)
)
""")

cursor.execute("""
SELECT p.PolicyID, p.StartDate, p.EndDate, p.CoverageType, p.Excess, v.Value
FROM Policy p
JOIN Vehicle v ON p.VehicleID = v.VehicleID
""")

policies = cursor.fetchall()

claim_types = [
    "Glass Damage","Vandalism","Accident(minor)",
    "Accident(major)","Weather","Theft","Fire"
]

def claim_probability():
    return random.random() < 0.18


def claim_amount(value, excess, damage):
    severity = {
        "Glass Damage": 0.08,
        "Vandalism": 0.15,
        "Accident(minor)": 0.25,
        "Accident(major)": 0.6,
        "Weather": 0.35,
        "Theft": 0.9,
        "Fire": 0.85
    }

    amount = value * severity[damage] - excess
    amount = max(0, amount)

    return round(amount, 2)


def claim_date(start, end):
    dt1 = datetime.strptime(str(start), "%Y-%m-%d")
    dt2 = datetime.strptime(str(end), "%Y-%m-%d")
    return (dt1 + timedelta(days=random.randint(0, (dt2 - dt1).days))).date()

claims_data = []

for policy_id, start, end, coverage, excess, value in policies:

    if not claim_probability():
        continue

    num_claims = random.choice([1, 1, 2])

    for _ in range(num_claims):

        damage = random.choice(claim_types)
        date_of_claim = claim_date(start, end)

        allowed = set(claim_types)

        if coverage == "Third party":
            allowed = {"Accident(minor)", "Accident(major)"}
        elif coverage == "Third party, fire and theft":
            allowed = {"Accident(minor)", "Accident(major)", "Theft", "Fire"}

        status = "Approved" if damage in allowed else "Denied"

        amount = 0 if status == "Denied" else claim_amount(value, excess, damage)

        claims_data.append((
            policy_id,
            date_of_claim,
            damage,
            status,
            amount
        ))

cursor.executemany("""
INSERT INTO Claims (PolicyID, ClaimDate, ClaimType, ClaimStatus, Amount)
VALUES (?, ?, ?, ?, ?)
""", claims_data)


# FINAL

conn.commit()
conn.close()

print("Insurance dataset generated successfully.")

Insurance dataset generated successfully.
